# WARIBA Offer Economics & Acquisition V1

Notebook de décision reproductible. Les faits externes sont documentés dans le rapport; toutes les variables non observées sont **HYPOTHÈSE** ou **CANDIDAT**.

In [1]:
from wariba_model import run_model
frames = run_model()
summary_cols = [
    'Scénario', 'Produit', 'CA_attendu_XOF', 'CAC_XOF', 'Payout_attendu_XOF',
    'Contribution_après_CAC_XOF', 'Contribution_margin_pct', 'LTV_sur_CAC',
    'Break_even_CAC_XOF', 'Réserve_requise_2_5x_XOF',
]
d = frames['detail']
country_stress = d[(d['Scénario'] == 'Stress') & (d['Taille_K'] == 25)][[
    'Produit', 'Pays', 'CAC_XOF', 'CA_attendu_XOF', 'Payout_attendu_XOF',
    'Contribution_après_CAC_XOF', 'Contribution_margin_pct', 'LTV_sur_CAC',
    'Break_even_CAC_XOF', 'Réserve_requise_2_5x_XOF',
]]

Modèle chargé et recalculé


## Scénarios portefeuille et produits

In [2]:
frames['scenario_summary'][summary_cols]

Scénario   Produit  CA_attendu_XOF   CAC_XOF  Payout_attendu_XOF  Contribution_après_CAC_XOF  Contribution_margin_pct  LTV_sur_CAC  Break_even_CAC_XOF  Réserve_requise_2_5x_XOF
    Base       ONE       60900.000 12100.000         2640.591360                40563.802726                 0.666072     4.352380        52663.802726               3630.813120
    Base      FLEX       29800.750  8470.000         1859.124960                15406.494790                 0.516983     2.818949        23876.494790               2556.296820
    Base   INSTANT       92400.000 18755.000         4185.043200                58853.306368                 0.636941     4.138006        77608.306368               5754.434400
    Base PORTFOLIO       45390.450 10587.500         2326.156704                27298.368329                 0.601412     3.578358        37885.868329               3198.465468
  Stress       ONE       60900.000 18755.000        13377.830400                18974.002144                 0.3115

## Stress tests

In [3]:
frames['stress_tests']

                          Test        CA  Contribution        CM                                Lecture
Stress candidat — mix souscrit 49688.688   8212.314447  0.165275                       PASS si CM >=15%
Stress + mix low-ticket 45% 5K 36642.896   1699.287488  0.046374    FAIL attendu: ne pas scaler 5K paid
              Stress + CAC 25% 49688.688   4109.658197  0.082708 Surveille la dérive auction/conversion
          Stress + payouts 25% 49688.688   4313.905929  0.086819              Surveille severity/cycles
 Disaster — résultats extrêmes 55900.740 -80232.154781 -1.435261       FAIL: circuit breaker et reserve


## Gating produit × taille

In [4]:
frames['size_gate']

Produit  Taille_K  CM_Stress_pct  Contribution_Stress_XOF Statut_lancement
    ONE         5      -0.397706             -7914.347360    NO PAID SCALE
    ONE        10       0.081914              2858.805280           LIMITÉ
    ONE        25       0.372364             26028.263200        GO PILOTE
    ONE        50       0.475555             57019.026400        GO PILOTE
    ONE       100       0.503755            100700.552800        GO PILOTE
   FLEX         5      -0.525960             -7757.105658    NO PAID SCALE
   FLEX        10      -0.180362             -4034.578515    NO PAID SCALE
   FLEX        25       0.161723              7354.066912        GO PILOTE
   FLEX        50       0.188426             13098.142624        GO PILOTE
   FLEX       100       0.045088              4302.574048           LIMITÉ
INSTANT         5      -0.203067             -8102.383280    NO PAID SCALE
INSTANT        10       0.077746              4656.983440           LIMITÉ
INSTANT        25       0

## Sensibilité des rules

In [5]:
frames['sensitivity'].sort_values('CM_Stress_pct', ascending=False).head(25)

 Target_ONE  Target_FLEX  Buffer_pct  Best_Day_pct     Leverage  Cap_scale  CM_Stress_pct  CM_Disaster_pct  Contribution_Stress_XOF  BE_CAC_Stress_XOF  Cap1_sur_prix_x  Appeal_score  Qualifie_garde_fous  Score_décision
         10            4         3.0            30 Conservateur        0.8       0.271964        -0.869863             13194.072886       29604.697886         4.158973         41.25                False        1.461446
         10            3         3.0            30 Conservateur        0.8       0.266695        -0.940695             13512.792615       29923.417615         4.158973         46.25                False        0.400867
         10            4         3.0            30 Conservateur        1.2       0.264430        -0.958354             12828.556570       29239.181570         6.238460         47.25                False        0.034779
         10            4         3.0            30 Conservateur        1.0       0.264430        -0.939352             12828

## Pays × CAC × unit economics — 25K Stress

In [6]:
country_stress

Produit          Pays  CAC_XOF  CA_attendu_XOF  Payout_attendu_XOF  Contribution_après_CAC_XOF  Contribution_margin_pct  LTV_sur_CAC  Break_even_CAC_XOF  Réserve_requise_2_5x_XOF
    ONE Côte d’Ivoire 17050.00        69900.00          14541.1200                27733.263200                 0.396756     2.626584        44783.263200                23629.3200
    ONE      Cameroun 18600.00        69900.00          14541.1200                26183.263200                 0.374582     2.407702        44783.263200                23629.3200
    ONE       Sénégal 23250.00        69900.00          14541.1200                21533.263200                 0.308058     1.926162        44783.263200                23629.3200
    ONE         Bénin 15500.00        69900.00          14541.1200                29283.263200                 0.418931     2.889243        44783.263200                23629.3200
    ONE  Burkina Faso 17825.00        69900.00          14541.1200                26958.263200           

## Contrôles de qualité

In [7]:
frames['checks']

                                Contrôle  Résultat  Attendu
                 Probabilités dans [0,1]      True     True
Contributions arithmétiques réconciliées      True     True
                Reserve 2,5x réconciliée      True     True
                          Aucun prix nul      True     True
             Stress portfolio CM positif      True     True
           Disaster portfolio CM positif     False     True
